# Task 1: train S3 (BERT) and S4 (RoBERTa) on Colab

Runtime → Change runtime type → **T4 GPU**. Run all cells in order.

1. Mount Drive (weights + results are saved there so they survive a disconnect).
2. Clone the repo branch and install pinned deps.
3. Re-create the exact Task 1 splits (checked against the local fingerprint).
4. Train S4 then S3 (max_length 256, fp16), evaluate each.


In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os
OUT = '/content/drive/MyDrive/steam-review-translator-task1'
os.makedirs(OUT, exist_ok=True)
os.environ['ARTIFACTS_DIR'] = OUT  # models -> OUT/models/task1, results -> OUT/runs/task1
!nvidia-smi --query-gpu=name,memory.total --format=csv

Mounted at /content/drive
name, memory.total [MiB]
Tesla T4, 15360 MiB


In [2]:
BRANCH = 'task1/data-and-eval'
!git clone -q --branch $BRANCH https://github.com/mahesa005/steam-review-translator.git /content/srt
%cd /content/srt
!pip install -q -e '.[task1]' pandas==3.1.1 numpy==2.5.5 pyarrow==26.0.1 py3langid==0.4.0 scikit-learn==1.9.1

/content/srt
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for steam-review-translator (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.1.1 which is incompatible.
numba 0.64.0 requires numpy<2.5,>=1.22, but you have numpy 2.5.5 which is incompatible.
cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.1.1 which is incompatible.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.1 which is incompatible.
dask-cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.1.1 which is incompatible.


In [3]:
# Same data as the local machine: download, prepare, and verify the fingerprint.
!make task1-download PYTHON=python 2>&1 | tail -2
!cd experiments/task1_sentiment && python prepare_metacritic.py 2>&1 | grep -v Warning | tail -3 && python prepare_steam.py --no-names 2>&1 | grep -v Warning | tail -2
out = !python experiments/task1_sentiment/fingerprint.py
print('\n'.join(out))
EXPECTED = {'metacritic.parquet': '0a8b9967d1189456', 'steam.parquet': '661e2fd1d614ab76'}
for name, h in EXPECTED.items():
    assert any(name in l and h in l for l in out), f'{name} differs from the local split: STOP and tell Farrel'
print('fingerprints match')

                                 Dload  Upload   Total   Spent    Left  Speed
100  482M  100  482M    0     0  1900k      0  0:04:19  0:04:19 --:--:-- 2427k
[460s] wrote 174,837 reviews / 200 games to /content/srt/data/processed/task1
{'train': 140, 'test': 30, 'val': 30} {'train': 124837, 'val': 25743, 'test': 24257}
labels: {'positive': 0.5879, 'negative': 0.277, 'neutral': 0.1351}
[474s] pass 2: 10,000 reviews
[474s] wrote 10,000 reviews / 50 apps to /content/srt/data/processed/task1
/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (
metacritic.parquet: rows=174,837 sha256=0a8b9967d1189456
steam.parquet: rows=10,000 sha256=661e2fd1d614ab76
fingerprints match


In [4]:
# S4: RoBERTa-base (main system)
!python experiments/task1_sentiment/train_transformer.py --model roberta-base --max-length 256 --fp16 --batch-size 32 --lr 2e-5 2>&1 | grep -vE 'Warning|warn'
!python experiments/task1_sentiment/evaluate.py --system transformer --name S4-roberta-len256 --model $OUT/models/task1/roberta-base_len256 --batch-size 64 2>&1 | grep -E '===|metacritic_test|steam |->'

  from pandas.core import (
device=cuda train=124,837 val=25,743 run=roberta-base_len256
class weights: {'positive': 0.566786527633667, 'negative': 1.2014878988265991, 'neutral': 2.479138135910034}
Loading weights: 100%|██████████| 197/197 [00:00<00:00, 4041.45it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missi

In [5]:
# S3: BERT-base
!python experiments/task1_sentiment/train_transformer.py --model bert-base-uncased --max-length 256 --fp16 --batch-size 32 --lr 2e-5 2>&1 | grep -vE 'Warning|warn'
!python experiments/task1_sentiment/evaluate.py --system transformer --name S3-bert-len256 --model $OUT/models/task1/bert-base-uncased_len256 --batch-size 64 2>&1 | grep -E '===|metacritic_test|steam |->'

  from pandas.core import (
device=cuda train=124,837 val=25,743 run=bert-base-uncased_len256
class weights: {'positive': 0.566786527633667, 'negative': 1.2014878988265991, 'neutral': 2.479138135910034}
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3742.04it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    |

In [6]:
# Summary of every finished run (copy this output back)
import json, glob
for f in sorted(glob.glob(f'{OUT}/models/task1/*/train_log.json')):
    print(f.split('/')[-2], 'best val macro-F1', round(json.load(open(f))['best_val_macro_f1'], 4))
for f in sorted(glob.glob(f'{OUT}/runs/task1/*/metrics.json')):
    m = json.load(open(f))
    for k in ('metacritic_test', 'steam'):
        r, d = m[k]['review'], m[k]['distribution']
        print(m['system'], k, 'acc', round(r['accuracy'], 3), 'macroF1', round(r['macro_f1'], 3), 'TVD', round(d['cc']['tvd'], 3), 'ACC-TVD', round(d['acc']['tvd'], 3))

bert-base-uncased_len256 best val macro-F1 0.7961
roberta-base_len256 best val macro-F1 0.8131
S3-bert-len256 metacritic_test acc 0.862 macroF1 0.783 TVD 0.035 ACC-TVD 0.052
S3-bert-len256 steam acc 0.885 macroF1 0.835 TVD 0.071 ACC-TVD 0.063
S4-roberta-len256 metacritic_test acc 0.877 macroF1 0.801 TVD 0.029 ACC-TVD 0.046
S4-roberta-len256 steam acc 0.892 macroF1 0.846 TVD 0.072 ACC-TVD 0.062


Optional: max_length 512 (about 2x slower). Pick the length by **validation** macro-F1 in `train_log.json`, then report test only for the chosen one.

In [7]:
# !python experiments/task1_sentiment/train_transformer.py --model roberta-base --max-length 512 --fp16 --batch-size 16 2>&1 | grep -vE 'Warning|warn'